Question 1: Cubic Histogram Classification Rule
Dataset: banknote_authentication.csv
Description: banknote_authentication.names
1. Select two variance features (Variance, Skewness)
2. Implement and visualize cubic histogram classification with:
a. Equal-width binning (3 bins/axis → 9 cells),
b. Max-count class assignment per cell.
3. Test accuracy with different bin configurations:
a. 2 × 2 bins → 4 cells
b. 4 × 4 bins → 16 cells
4. Analyze:
a. how bin granularity affects overfitting?
b. why this method is impractical for high-dimensional data?

In [17]:
import pandas as pd
import numpy as np

from glob import glob

import IPython.display as ipd
from itertools import cycle

import os
import io
from pydantic import BaseModel, computed_field, Field
from typing import Any, cast
from tqdm import tqdm
import plotly.express as px
import plotly.graph_objects as go
import plotly.colors as pc
from plotly.subplots import make_subplots
from collections import Counter


color_palette = pc.qualitative.Plotly  # or try: D3, G10, T10, Pastel, Bold
color_cycle = cycle(color_palette)

# data foler from current file location
DATA_DIR = os.path.join(os.getcwd(), "../data")
RAW_DIR = os.path.join(DATA_DIR, "raw")
DATASETT_NAME = "banknote_authentication.csv"
DATASETT_PATH = os.path.join(RAW_DIR, DATASETT_NAME)
DATASETT_PATH

'/Users/valiantlynx/projects/pattern-recognition/notebooks/../data/raw/banknote_authentication.csv'

In [18]:
# --- Load & label data ---
df = pd.read_csv(DATASETT_PATH, names=["variance", "skeweness", "curtosis", "entropy", "class"])

feat_x = "variance"
feat_y = "skeweness"
X = df[feat_x].values
Y = df[feat_y].values
labels = df["class"].values

N_BINS = 3

# --- Equal-width binning ---
x_edges = np.linspace(X.min(), X.max(), N_BINS + 1)
y_edges = np.linspace(Y.min(), Y.max(), N_BINS + 1)

x_bin = np.clip(np.digitize(X, x_edges) - 1, 0, N_BINS - 1)
y_bin = np.clip(np.digitize(Y, y_edges) - 1, 0, N_BINS - 1)

# --- Max-count class assignment per cell ---
cell_class = {}
cell_counts = {}

for i in range(N_BINS):
    for j in range(N_BINS):
        mask = (x_bin == i) & (y_bin == j)
        points_in_cell = labels[mask]
        if len(points_in_cell) > 0:
            majority = Counter(points_in_cell).most_common(1)[0][0]
            cell_class[(i, j)] = majority
            cell_counts[(i, j)] = Counter(points_in_cell)
        else:
            cell_class[(i, j)] = None

# --- Build Plotly figure ---
fig = go.Figure()

cell_colors = {0: "rgba(174, 214, 241, 0.5)", 1: "rgba(169, 223, 171, 0.5)", None: "rgba(240,240,240,0.3)"}

for i in range(N_BINS):
    for j in range(N_BINS):
        x0, x1 = x_edges[i], x_edges[i + 1]
        y0, y1 = y_edges[j], y_edges[j + 1]
        cls = cell_class[(i, j)]
        counts = cell_counts.get((i, j), {})
        count_str = "<br>".join([f"Class {k}: {v}" for k, v in sorted(counts.items())])
        hover = f"Cell ({i},{j})<br>Assigned: Class {cls}<br>{count_str}"

        # Filled rectangle as a scatter shape
        fig.add_shape(
            type="rect",
            x0=x0, x1=x1, y0=y0, y1=y1,
            fillcolor=cell_colors[cls],
            line=dict(color="black", width=1.5),
        )

        # Cell annotation
        label = f"→ Class {cls}<br>{count_str}" if cls is not None else "Empty"
        fig.add_annotation(
            x=(x0 + x1) / 2,
            y=(y0 + y1) / 2,
            text=label,
            showarrow=False,
            font=dict(size=10),
            align="center",
        )

        # Invisible scatter just for hover on cells
        fig.add_trace(go.Scatter(
            x=[(x0 + x1) / 2],
            y=[(y0 + y1) / 2],
            mode="markers",
            marker=dict(size=0.1, opacity=0),
            hovertemplate=hover + "<extra></extra>",
            showlegend=False,
        ))

# --- Scatter actual data points ---
class_styles = {
    0: dict(color="blue", name="Class 0 (Fake)"),
    1: dict(color="green", name="Class 1 (Real)"),
}

for cls, style in class_styles.items():
    mask = labels == cls
    fig.add_trace(go.Scatter(
        x=X[mask],
        y=Y[mask],
        mode="markers",
        marker=dict(color=style["color"], size=5, opacity=0.5),
        name=style["name"],
    ))

fig.update_layout(
    title="Cubic Histogram Classifier (3×3 grid)",
    xaxis_title=feat_x,
    yaxis_title=feat_y,
    legend=dict(x=1.02, y=1),
    width=800,
    height=700,
)

fig.show()

In [19]:
class BankNote(BaseModel):
    variance_of_wavelet_transformed_image_continuous: float = Field(alias='variance')
    skeweness_of_wavelet_transformed_image_continuos: float = Field(alias="skeweness")
    curtosis_of_wavelet_transformed_image_continuos: float = Field(alias="curtosis")
    entropy_of_image_continuos: float = Field(alias="entropy")
    true_class: int = Field(alias="class")

banknotes: list[BankNote]  = []


In [20]:
banknotes_df = pd.read_csv(
    DATASETT_PATH,
    names=["variance", "skeweness", "curtosis", "entropy", "class"]
)
banknotes = [BankNote(**row) for row in banknotes_df.to_dict(orient="records")]
banknotes_df

,variance,skeweness,curtosis,entropy,class
0,3.62160,8.66610,-2.8073,-0.44699,0
1,4.54590,8.16740,-2.4586,-1.46210,0
2,3.86600,-2.63830,1.9242,0.10645,0
3,3.45660,9.52280,-4.0112,-3.59440,0
4,0.32924,-4.45520,4.5718,-0.98880,0
...,...,...,...,...,...
1367,0.40614,1.34920,-1.4501,-0.55949,1
1368,-1.38870,-4.87730,6.4774,0.34179,1
1369,-3.75030,-13.45860,17.5932,-2.77710,1
1370,-3.56370,-8.38270,12.3930,-1.28230,1


In [21]:

X = banknotes_df["variance"].values
Y = banknotes_df["skeweness"].values
labels = banknotes_df["class"].values
N_BINS: int = 3
print("X_shape: %s and head: %s" % (X.shape, X[:10]))
print("Y_shape: %s and head: %s" % (Y.shape, Y[:10]))

X_shape: (1372,) and head: [3.6216  4.5459  3.866   3.4566  0.32924 4.3684  3.5912  2.0922  3.2032
 1.5356 ]
Y_shape: (1372,) and head: [ 8.6661  8.1674 -2.6383  9.5228 -4.4552  9.6718  3.0129 -6.81    5.7588
  9.1772]


### Get the borders of the buckets
since its 3 buckets in a list (2D) 
```sh
edge[0]         edge[1]          edge[2]          edge[3]
|------bin_0------|------bin_1------|------bin_2------|
```

In [22]:
X = np.asarray(X) # for typing issues, turn it to normal array from np.aarray
x_max = cast(float, X.max()) # this is basicly a "Trust me bro, i know this is a float"
x_min = cast(float, X.min())

x_edges = np.linspace(x_min, x_max, int(N_BINS) + 1)
x_edges

array([-7.0421, -2.4198,  2.2025,  6.8248])

In [23]:
Y = np.asarray(Y)
y_max = cast(float, Y.max())
y_min = cast(float, Y.min())

y_edges = np.linspace(y_min, y_max, int(N_BINS) + 1)
y_edges

array([-13.7731    ,  -4.86486667,   4.04336667,  12.9516    ])

### actually make the bins

In [24]:
x_bin = np.clip(np.digitize(X, x_edges) - 1, 0, 2) # force the bins to the clossest bin if it was in the boundary(something that np.digitilize does sometimes putting stuff in the phantom bin)
x_bin

array([2, 2, 2, ..., 0, 0, 0], shape=(1372,))

In [25]:
y_bin = np.clip(np.digitize(Y, y_edges), 0, 2)
y_bin

array([2, 2, 2, ..., 1, 1, 2], shape=(1372,))

In [26]:

fig = go.Figure()

fig.add_trace(go.Scatter(
    y=X,
    mode="markers",
    name="Variation",
    marker=dict(color=color_palette[4], opacity=0.5)
))

fig.add_trace(go.Scatter(
    y=Y,
    mode="markers",
    name="Skeweness",
    marker=dict(color=color_palette[6], symbol="diamond")
))

fig.update_layout(
    title="Variance and Skeweness - just to see how it would look",
    xaxis_title="Index",
    yaxis_title="Value"
)

fig.show()

In [35]:
cell_class = {}
cell_counts_in_the_grid = {}

for i in range(N_BINS):
    for j in range(N_BINS):
        mask = (x_bin == i) & (y_bin == j)
        points_in_cell = labels[mask]
        count = Counter(points_in_cell)
        if len(count) > 0:
            majority = count.most_common(1)[0][0]
            cell_class[(i,j)] = majority
            cell_counts[(i, j)] = Counter(points_in_cell)
        else:
            cell_counts[(i, j)] = None
            
print(cell_class)
print(len(cell_counts_in_the_grid))


{(0, 1): np.int64(1), (0, 2): np.int64(1), (1, 1): np.int64(1), (1, 2): np.int64(0), (2, 1): np.int64(0), (2, 2): np.int64(0)}
0
